In [50]:
import pandas as pd
import numpy as np

In [51]:
file_path = 'SIN TOCAR - transactions_full_with_dirty_records.csv'

df = pd.read_csv(file_path)

# Visualizacion de las dimensiones iniciales
print(df.shape)

# Visualizacion de las primeras filas
print(df.head())

(600000, 31)
  transaction_id customer_id transaction_datetime transaction_date  \
0    TX000442322   CUST04737  2025-01-16 06:18:07       2025-01-16   
1    TX000426517   CUST05210  2025-03-14 20:02:51       2025-03-14   
2    TX000215101   CUST01678  2025-01-06 13:49:10       2025-01-06   
3    TX000478746   CUST01387  2025-01-11 21:31:46       2025-01-11   
4    TX000489506   CUST04801  2025-01-27 13:34:17       2025-01-27   

  transaction_time                   product_type transaction_amount  \
0         06:18:07                         ahorro             3258.3   
1         20:02:51                pago de tarjeta            2324.31   
2         13:49:10                         ahorro            3905.44   
3         21:31:46  transferencia cuenta a cuenta            1272.11   
4         13:34:17  transferencia cuenta a cuenta             3073.1   

  transaction_direction   channel transaction_status  ...  \
0                salida  sucursal         completada  ...   
1          

In [52]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Filas: 600000
Columnas: 31


In [53]:
print("\nCOLUMNAS:")
print(df.columns.tolist())



COLUMNAS:
['transaction_id', 'customer_id', 'transaction_datetime', 'transaction_date', 'transaction_time', 'product_type', 'transaction_amount', 'transaction_direction', 'channel', 'transaction_status', 'merchant_or_destination', 'source_account_type', 'destination_account_type', 'customer_segment', 'occupation', 'industry', 'employment_type', 'monthly_income', 'income_frequency', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count', 'usual_transaction_hour_range', 'usual_product_type', 'usual_channel', 'deviation_from_usual_amount_pct', 'anomaly_flag', 'risk_profile', 'risk_score', 'fraud_label', 'data_quality_flag']


In [54]:
print("\nTIPOS DE DATOS:")
print(df.dtypes)


TIPOS DE DATOS:
transaction_id                      object
customer_id                         object
transaction_datetime                object
transaction_date                    object
transaction_time                    object
product_type                        object
transaction_amount                  object
transaction_direction               object
channel                             object
transaction_status                  object
merchant_or_destination             object
source_account_type                 object
destination_account_type            object
customer_segment                    object
occupation                          object
industry                            object
employment_type                     object
monthly_income                     float64
income_frequency                    object
customer_tenure_months               int64
avg_3m_transaction_amount          float64
avg_3m_transaction_count             int64
usual_transaction_hour_range        o

In [55]:
print("\nNULOS POR COLUMNA:")
print(df.isnull().sum())


NULOS POR COLUMNA:
transaction_id                      1729
customer_id                        15594
transaction_datetime               13654
transaction_date                   13477
transaction_time                       0
product_type                       13996
transaction_amount                 13829
transaction_direction                  0
channel                             2094
transaction_status                  2503
merchant_or_destination                0
source_account_type                    0
destination_account_type               0
customer_segment                       0
occupation                         13814
industry                               0
employment_type                        0
monthly_income                     13737
income_frequency                       0
customer_tenure_months                 0
avg_3m_transaction_amount              0
avg_3m_transaction_count               0
usual_transaction_hour_range           0
usual_product_type                   

In [56]:
# Conteo de filas completamente duplicadas
full_duplicates = df.duplicated().sum()

print(f"Duplicados completos encontrados: {full_duplicates}")

Duplicados completos encontrados: 4309


#### 1. Verificación de filas completamente duplicadas:

Se verificó si existían filas idénticas en todas las columnas.

In [57]:
# Busqueda por transaction_id duplicados
transaction_duplicates = df[df.duplicated(subset=['transaction_id'], keep=False)]

print(transaction_duplicates.shape)

(172406, 31)


#### 2. Verificación de duplicados por transaction_id

El campo más crítico normalmente es el transaction_id.

Si un mismo transaction_id aparece varias veces con exactamente la misma información, esto puede indicar:

- Error de inserción.
- Duplicación durante ETL.
- Problemas de sincronización.
- Replicación accidental.


En datasets financieros, los duplicados deben manejarse con extremo cuidado, ya que:

Una transacción legítima puede parecer repetida.
Un cliente puede realizar varias operaciones iguales en segundos.
Eliminar registros incorrectamente puede afectar análisis financieros, métricas y modelos de Machine Learning.
Los duplicados reales pueden inflar montos, alterar conteos y generar sesgos en la detección de fraude.

Por esta razón, antes de eliminar registros, se realizó una validación detallada para diferenciar:

- Transacciones legítimas similares.
- Duplicados exactos.
- Duplicados parciales.
- Posibles errores de carga o replicación.

In [58]:
# ejemplos
transaction_duplicates.sort_values('transaction_id').head(10)

,transaction_id,customer_id,transaction_datetime,transaction_date,transaction_time,product_type,transaction_amount,transaction_direction,channel,transaction_status,...,avg_3m_transaction_count,usual_transaction_hour_range,usual_product_type,usual_channel,deviation_from_usual_amount_pct,anomaly_flag,risk_profile,risk_score,fraud_label,data_quality_flag
36847,123,UNKNOWN,2025-03-18 09:11:41,2025-03-18,09:11:41,transferencia cuenta a cuenta,6822.06,entrada,web,completada,...,76,mañana,transferencia cuenta a cuenta,web,24.82,0,bajo,474,0,invalid_format
367308,123,NaN,2025-02-22 18:27:49,2025-02-22,18:27:49,transferencia cuenta a cuenta,27894.11,salida,ATM,pendiente,...,74,tarde,transferencia cuenta a cuenta,ATM,2762.28,1,alto,750,1,invalid_format
416129,123,UNKNOWN,2025-02-13 19:12:52,2025-02-13,19:12:52,pago de tarjeta,1726.32,salida,ATM,completada,...,85,noche,transferencia cuenta a cuenta,app,10.66,0,medio,508,0,invalid_format
248176,123,NaN,2025-02-17 21:18:32,2025-02-17,21:18:32,transferencia cuenta a cuenta,13767.43,salida,sucursal,completada,...,78,noche,pago de prestamo,app,30.56,0,bajo,361,0,invalid_format
274101,123,CUST?,2025-03-29 22:47:13,2025-03-29,22:47:13,ahorro,16306.19,salida,ATM,completada,...,142,noche,ahorro,app,42.47,0,medio,553,0,invalid_format
523416,123,UNKNOWN,2025-03-03 06:27:28,2025-03-03,06:27:28,pago de tarjeta,3076.82,salida,app,completada,...,65,mañana,transferencia cuenta a cuenta,app,18.62,0,bajo,483,0,invalid_format
523401,123,CUST?,2025-01-02 17:16:09,2025-01-02,17:16:09,transferencia cuenta a cuenta,6017.88,salida,agente,completada,...,68,tarde,ahorro,app,31.24,0,bajo,403,0,invalid_format
367151,123,NaN,2025-03-23 19:30:47,2025-03-23,19:30:47,ahorro,1916.91,salida,sucursal,completada,...,136,noche,ahorro,sucursal,-51.93,0,medio,552,0,invalid_format
79030,123,NaN,2025-03-10 17:26:37,2025-03-10,17:26:37,transferencia cuenta a cuenta,5765.28,salida,ATM,completada,...,133,tarde,ahorro,ATM,97.35,0,bajo,494,0,invalid_format
248216,123,NaN,2025-01-11 08:05:37,2025-01-11,08:05:37,transferencia cuenta a cuenta,813.79,salida,web,completada,...,21,mañana,ahorro,app,-2.25,0,bajo,443,0,invalid_format


#### 3. Visualización de registros duplicados

Esta inspeccion manual permitió validar:

- Si las filas eran exactamente iguales.
- Si existían diferencias entre columnas.
- Si el duplicado representaba una transacción legítima.

In [59]:
# cantidad antes de limpiar
rows_before = df.shape[0]

# Eliminacion de duplicados exactos

df = df.drop_duplicates()

# Cantidad después
rows_after = df.shape[0]

# Registros eliminados
removed_rows = rows_before - rows_after

print(f"Registros eliminados: {removed_rows}")

Registros eliminados: 4309


#### Estrategia de Limpieza Aplicada
Consideraciones importantes:

Debido a que el dataset pertenece al dominio financiero, NO se eliminaron automáticamente todos los registros similares.

Decisiones:
Filas completamente idénticas:	Eliminar
Mismo transaction_id + misma información: Eliminar
Transacciones similares pero con distinto ID: Mantener
Transacciones con mismo monto/hora/cliente: Mantener hasta validar

In [60]:
# Verificando nuevamente
print(df.duplicated().sum())

0


In [61]:
# Validacion de Transaction IDs duplicados restantes
remaining_duplicates = df['transaction_id'].duplicated().sum()

print(f"transaction_id duplicados restantes: {remaining_duplicates}")

transaction_id duplicados restantes: 95688


#### Conclusiones
Durante el proceso de limpieza se identificaron registros duplicados dentro del dataset financiero.

Debido a la naturaleza sensible de las transacciones, no se eliminaron registros únicamente por similitud de valores, ya que múltiples transacciones legítimas pueden compartir:

- Monto.
- Cliente.
- Canal.
- Hora.
- Tipo de producto.

La limpieza se enfocó exclusivamente en:

- Filas completamente idénticas.
- Duplicados exactos de transaction_id.

Se aplicó drop_duplicates() únicamente para eliminar registros repetidos reales y preservar la integridad histórica del dataset.

Esta decisión ayuda a:

- Mantener transacciones válidas.
- Evitar pérdida de información.
- Reducir sesgos analíticos.
- Mejorar la calidad de modelos de Machine Learning.
- Garantizar mayor confiabilidad en análisis de fraude financiero.

In [62]:
import unicodedata #Adison

print("HOMOGENIZANDO COLUMNAS DE TEXTO")

text_columns = [
    "product_type",
    "transaction_direction",
    "channel",
    "transaction_status",
    "customer_segment",
    "occupation",
    "industry",
    "employment_type",
    "income_frequency",
    "usual_product_type",
    "usual_channel",
    "risk_profile",
    "data_quality_flag"
]


HOMOGENIZANDO COLUMNAS DE TEXTO


In [63]:

# Función auxiliar para remover acentos de forma segura
def remover_acentos(texto):
    if pd.isna(texto) or texto == 'nan':
        return texto
    # Normaliza el texto para separar las letras de sus acentos y filtra los caracteres de acentuación
    return "".join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

for col in text_columns:
    if col in df.columns:
        #  Convertir a string, remover espacios en blanco extremos y pasar a minúsculas
        df[col] = df[col].astype(str).str.strip().str.lower()
        
        #  Remover acentos y tildes para estandarizar la codificación
        df[col] = df[col].apply(remover_acentos)
        
        #  Corregir efectos secundarios de la conversión a string en datos faltantes
        # Si quedó como cadena vacía "" o como "nan", lo estandarizamos a "unknown"
        df[col] = df[col].replace(["", "nan"], "unknown")

print("¡Homogeneización de texto completada con éxito!")

¡Homogeneización de texto completada con éxito!


#### Proceso de Homogeneización de Texto (Adison)

La homogeneización de texto se realizó con el objetivo de estandarizar las variables categóricas y reducir inconsistencias semánticas dentro del dataset financiero. Para ello, se aplicaron técnicas de limpieza utilizando:

.str.strip() para eliminar espacios en blanco al inicio y final de los textos.
.str.lower() para convertir todos los valores a minúsculas.
Eliminación de acentos y caracteres especiales.
Reemplazo de cadenas vacías, valores nulos y textos como "nan" por "unknown".

Este proceso fue necesario porque en bases de datos financieras integradas desde múltiples fuentes es común encontrar variaciones de escritura para una misma categoría, por ejemplo:

- "Crédito"
- "credito"
- "credito "
- "CREDITO"

Aunque representan el mismo concepto, una computadora los interpreta como categorías diferentes, generando fragmentación y ruido en los datos.

La estandarización textual permitió:

- Reducir la creación de categorías duplicadas o falsas.
- Mejorar la consistencia semántica del dataset.
- Facilitar los procesos posteriores de codificación (encoding).
- Evitar la generación innecesaria de columnas redundantes.
- Mejorar la calidad analítica y el desempeño de modelos de Machine Learning.

Además, esta limpieza ayuda a que los algoritmos identifiquen patrones reales de comportamiento financiero y reduzcan errores derivados de inconsistencias textuales.

In [64]:
print("CONVIRTIENDO FECHAS")

date_columns = [
    "transaction_datetime",
    "transaction_date"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

#antes_fecha = df.shape[0]

# eliminar fechas inválidas
#df = df.dropna(subset=["transaction_datetime"])

#despues_fecha = df.shape[0]

#print(f"Filas eliminadas por fechas inválidas: {antes_fecha - despues_fecha}")


CONVIRTIENDO FECHAS


## Fechas Invalidas:
### Consideramos importante mantener las transacciones con fechas "invalidas", porque representan aun asi transacciones existentes. 
### Fechas duplicadas podrian representar, movimientos adicionales reales.
### Desicion: En base al criterio y naturaleza del dataset, preservaremos las transacciones, y realizar en su lugar una estandarizacion de las fechas y horas para cada transaccion.

In [65]:
df["transaction_datetime"] = pd.to_datetime(df['transaction_datetime'],errors="coerce")

fechas_invalidas = df[df['transaction_datetime'].isnull()]

print(fechas_invalidas.shape)


(16220, 31)


### 2. Proceso de ajuste/corrección de fechas/horas (Daniel)

In [77]:
print("DIAGNÓSTICO — ESTADO ACTUAL DE COLUMNAS DE FECHAS/HORAS")
print(f"Nulos en transaction_datetime : {df['transaction_datetime'].isnull().sum():>10,}")
print(f"Nulos en transaction_date     : {df['transaction_date'].isnull().sum():>10,}")
print(f"Nulos en transaction_time     : {df['transaction_time'].isnull().sum():>10,}")
print(f"\nTipo transaction_datetime : {df['transaction_datetime'].dtype}")
print(f"Tipo transaction_date     : {df['transaction_date'].dtype}")
print(f"Tipo transaction_time     : {df['transaction_time'].dtype}")
print(df[["transaction_time"]].head())

DIAGNÓSTICO — ESTADO ACTUAL DE COLUMNAS DE FECHAS/HORAS
Nulos en transaction_datetime :      6,050
Nulos en transaction_date     :      6,050
Nulos en transaction_time     :          0

Tipo transaction_datetime : datetime64[ns]
Tipo transaction_date     : datetime64[ns]
Tipo transaction_time     : object
  transaction_time
0         06:18:07
1         20:02:51
2         13:49:10
3         21:31:46
4         13:34:17


In [67]:
import re

def estandarizar_hora(valor):
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip()
    if re.fullmatch(r'\d{1,2}:\d{2}:\d{2}', s):
        h, m, seg = map(int, s.split(':'))
        if 0 <= h <= 23 and 0 <= m <= 59 and 0 <= seg <= 59:
            return f"{h:02d}:{m:02d}:{seg:02d}"
    if re.fullmatch(r'\d{1,2}:\d{2}', s):
        h, m = map(int, s.split(':'))
        if 0 <= h <= 23 and 0 <= m <= 59:
            return f"{h:02d}:{m:02d}:00"
    return np.nan

invalidos_antes = df['transaction_time'].isnull().sum()
df['transaction_time'] = df['transaction_time'].apply(estandarizar_hora)
invalidos_despues = df['transaction_time'].isnull().sum()

print(f"transaction_time inválidos antes : {invalidos_antes}")
print(f"transaction_time inválidos después: {invalidos_despues}")
print(f"Formatos inválidos detectados    : {invalidos_despues - invalidos_antes}")

transaction_time inválidos antes : 0
transaction_time inválidos después: 3809
Formatos inválidos detectados    : 3809


In [68]:
nulos_antes = df['transaction_datetime'].isnull().sum()

mask_recover = df['transaction_datetime'].isnull() & df['transaction_date'].notnull()
hora_str  = df.loc[mask_recover, 'transaction_time'].fillna('00:00:00')
fecha_str = df.loc[mask_recover, 'transaction_date'].dt.strftime('%Y-%m-%d')

df.loc[mask_recover, 'transaction_datetime'] = pd.to_datetime(fecha_str + ' ' + hora_str, errors='coerce')

nulos_despues = df['transaction_datetime'].isnull().sum()
print(f"Nulos en transaction_datetime antes   : {nulos_antes:,}")
print(f"Registros recuperados                 : {nulos_antes - nulos_despues:,}")
print(f"Nulos en transaction_datetime después : {nulos_despues:,}")

Nulos en transaction_datetime antes   : 16,220
Registros recuperados                 : 10,170
Nulos en transaction_datetime después : 6,050


In [69]:
nulos_antes_date = df['transaction_date'].isnull().sum()

mask_date = df['transaction_date'].isnull() & df['transaction_datetime'].notnull()
df.loc[mask_date, 'transaction_date'] = df.loc[mask_date, 'transaction_datetime'].dt.normalize()

nulos_despues_date = df['transaction_date'].isnull().sum()
print(f"Nulos en transaction_date antes  : {nulos_antes_date:,}")
print(f"Registros recuperados            : {nulos_antes_date - nulos_despues_date:,}")
print(f"Nulos en transaction_date después: {nulos_despues_date:,}")

Nulos en transaction_date antes  : 16,131
Registros recuperados            : 10,081
Nulos en transaction_date después: 6,050


In [ ]:
FECHA_INICIO = pd.Timestamp('2025-01-01')
FECHA_FIN    = pd.Timestamp('2025-03-31 23:59:59')

mask_valido  = df['transaction_datetime'].notna()
mask_antes   = mask_valido & (df['transaction_datetime'] < FECHA_INICIO)
mask_despues = mask_valido & (df['transaction_datetime'] > FECHA_FIN)
mask_fuera   = mask_antes | mask_despues

mask_actualizar = mask_fuera & (df['data_quality_flag'] == 'valid')

print(f"Fechas anteriores al 2025-01-01 : {mask_antes.sum():,}")
print(f"Fechas posteriores al 2025-03-31: {mask_despues.sum():,}")
print(f"Total fuera del rango valido    : {mask_fuera.sum():,}")
print(f"Registros actualizados en flag  : {mask_actualizar.sum():,}")

df.loc[mask_actualizar, 'data_quality_flag'] = 'invalid_format'

print("\nSolo se actualizo data_quality_flag en registros que eran 'valid'.")
print("Nota: no se eliminaron filas.")

In [81]:
mask_dt_valido = df['transaction_datetime'].notna()
df.loc[mask_dt_valido, 'transaction_date'] = df.loc[mask_dt_valido, 'transaction_datetime'].dt.normalize()

mask_time_nulo = mask_dt_valido & df['transaction_time'].isnull()
df.loc[mask_time_nulo, 'transaction_time'] = df.loc[mask_time_nulo, 'transaction_datetime'].dt.strftime('%H:%M:%S')
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S', errors='coerce').dt.time

print("=" * 55)
print("RESUMEN FINAL — COLUMNAS DE FECHAS/HORAS")
print("=" * 55)
print(f"Nulos en transaction_datetime : {df['transaction_datetime'].isnull().sum():,}")
print(f"Nulos en transaction_date     : {df['transaction_date'].isnull().sum():,}")
print(f"Nulos en transaction_time     : {df['transaction_time'].isnull().sum():,}")
print(f"\nFecha mínima registrada: {df['transaction_datetime'].min()}")
print(f"Fecha máxima registrada: {df['transaction_datetime'].max()}")
print(f"Total de filas en el dataset : {df.shape[0]:,}")
print(df[["transaction_time"]].head())
print(df['transaction_time'].dropna().head(3).tolist())

RESUMEN FINAL — COLUMNAS DE FECHAS/HORAS
Nulos en transaction_datetime : 6,050
Nulos en transaction_date     : 6,050
Nulos en transaction_time     : 0

Fecha mínima registrada: 2025-01-01 06:00:00
Fecha máxima registrada: 2025-03-31 22:59:57
Total de filas en el dataset : 593,963
  transaction_time
0         06:18:07
1         20:02:51
2         13:49:10
3         21:31:46
4         13:34:17
[datetime.time(6, 18, 7), datetime.time(20, 2, 51), datetime.time(13, 49, 10)]


#### Proceso de ajuste/corrección de fechas/horas (Daniel)

El proceso de ajuste de fechas y horas se realizó con el objetivo de estandarizar y recuperar los datos temporales del dataset financiero sin eliminar registros. Dado que cada fila representa una transacción con valor económico, se priorizó la recuperación sobre el descarte.

**Cómo se hizo:**
Se aplicaron cuatro pasos secuenciales. Primero, se validó y estandarizó `transaction_time` al formato `HH:MM:SS`, marcando como `NaN` los valores con horas, minutos o segundos fuera de rango (ej. `25:00:00`). Segundo, se reconstruyó `transaction_datetime` combinando `transaction_date` + `transaction_time` en los registros donde el datetime era nulo pero las columnas individuales estaban disponibles. Tercero, se derivó `transaction_date` desde `transaction_datetime` para los registros con fecha nula. Finalmente, se validó el rango temporal permitido (2025-01-01 a 2025-03-31) y los registros fuera de rango fueron marcados en `data_quality_flag` como `invalid_format` sin ser eliminados.

**Por qué se hizo:**
Las columnas `transaction_datetime`, `transaction_date` y `transaction_time` son críticas para análisis de comportamiento del cliente y detección de fraude. Un registro sin fecha válida pierde trazabilidad, pero uno con fecha reconstruida sigue aportando información útil. En datasets financieros, eliminar registros implica pérdida de información sobre movimientos reales de dinero.

**Para qué sirve:**
Garantizar la integridad temporal permite que los modelos de Machine Learning identifiquen correctamente patrones de uso por hora, día y mes. Una fecha correcta es esencial para validar si una transacción ocurrió dentro del comportamiento habitual del cliente y para calcular métricas de detección de anomalías y fraude con mayor precisión.

In [73]:
print("MANEJO DE VALORES NULOS") #Adison

# 1. Identificar nulos en el ID de transacción antes de operar
antes_nulos_id = df.shape[0]
# El ID de transacción es crítico; si no existe, la transacción no es rastreable
df = df.dropna(subset=["transaction_id"])
print(f"Filas eliminadas por falta de transaction_id: {antes_nulos_id - df.shape[0]}")

# 2. Imputación de variables numéricas
# Para 'transaction_amount', si falta el monto, usamos la mediana de los montos de transacciones
if "transaction_amount" in df.columns:
    # Primero nos aseguramos de que sea de tipo numérico (por si viene como texto en el CSV original)
    df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors="coerce")
    mediana_monto = df["transaction_amount"].median()
    df["transaction_amount"] = df["transaction_amount"].fillna(mediana_monto)

# Para 'monthly_income', imputamos con la mediana para no sesgar por valores atípicos extremos
if "monthly_income" in df.columns:
    mediana_ingreso = df["monthly_income"].median()
    df["monthly_income"] = df["monthly_income"].fillna(mediana_ingreso)

# 2. Imputación de variables numéricas
# Para 'transaction_amount', si falta el monto, usamos la mediana de los montos de transacciones
if "transaction_amount" in df.columns:
    # Primero nos aseguramos de que sea de tipo numérico (por si viene como texto en el CSV original)
    df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors="coerce")
    mediana_monto = df["transaction_amount"].median()
    df["transaction_amount"] = df["transaction_amount"].fillna(mediana_monto)

# Para 'monthly_income', imputamos con la mediana para no sesgar por valores atípicos extremos
if "monthly_income" in df.columns:
    mediana_ingreso = df["monthly_income"].median()
    df["monthly_income"] = df["monthly_income"].fillna(mediana_ingreso)

# 3. Imputación de variables categóricas (Texto)
# Reemplazar nulos por una etiqueta explícita de "desconocido" o "no especificado"
categoricas_con_nulos = ["customer_id", "product_type", "occupation", "risk_profile", "channel", "transaction_status"]

for col in categoricas_con_nulos:
    if col in df.columns:
        # Nota: Como en el paso anterior convertiste todo a 'str', los nulos podrían haberse convertido en el texto 'nan'.
        # Manejamos ambos casos para asegurar la limpieza:
        df[col] = df[col].replace("nan", np.nan)
        df[col] = df[col].fillna("unknown")

print(f"Cantidad de nulos remanentes en el dataset: {df.isnull().sum().sum()}")



MANEJO DE VALORES NULOS
Filas eliminadas por falta de transaction_id: 1728
Cantidad de nulos remanentes en el dataset: 12100


#### Proceso de Manejo de Datos Nulos (Adison)

El manejo de datos nulos se realizó para preservar la calidad e integridad del dataset financiero sin perder información importante. Primero, se eliminaron los registros sin transaction_id, ya que este campo es esencial para garantizar la trazabilidad de las transacciones.

Posteriormente, se aplicó una estrategia de imputación según el tipo de variable:

Las variables numéricas, como transaction_amount, se rellenaron con la mediana para evitar el sesgo provocado por valores atípicos frecuentes en datos financieros.
Las variables categóricas, como occupation o risk_profile, se completaron con la etiqueta "unknown".

Esta estrategia permitió conservar la mayor cantidad posible de registros válidos, evitando una reducción significativa de las 600,000 filas originales del dataset. Además, ayudó a mantener una estructura más consistente y adecuada para los modelos de Machine Learning orientados a la detección de fraude.

In [74]:
# ============================================================
# 5. PROCESO DE MANEJO / AJUSTE DE COLUMNAS NUMÉRICAS (RONI)
# ============================================================

print("\n" + "=" * 70)
print("5. MANEJO / AJUSTE DE COLUMNAS NUMÉRICAS - RONI")
print("=" * 70)

# Validación de seguridad: asegura que el DataFrame exista antes de continuar.
if "df" not in globals():
    raise NameError("El DataFrame 'df' no está definido. Ejecuta primero la celda de carga del dataset.")

print(f"Dimensiones actuales del dataset: {df.shape}")

# 1. Columnas numéricas esperadas según la estructura del dataset financiero.
columnas_numericas_esperadas = [
    "transaction_amount",
    "monthly_income",
    "customer_tenure_months",
    "avg_3m_transaction_amount",
    "avg_3m_transaction_count",
    "deviation_from_usual_amount_pct",
    "risk_score",
    "anomaly_flag",
    "fraud_label"
]

# 2. Seleccionar solamente las columnas que realmente existen en el dataset.
columnas_numericas = [col for col in columnas_numericas_esperadas if col in df.columns]
columnas_no_encontradas = [col for col in columnas_numericas_esperadas if col not in df.columns]

print("\nColumnas numéricas encontradas:")
print(columnas_numericas)

if columnas_no_encontradas:
    print("\nColumnas esperadas que no fueron encontradas en el dataset:")
    print(columnas_no_encontradas)

# Si ninguna columna esperada existe, se detiene el proceso con un mensaje claro.
if len(columnas_numericas) == 0:
    raise ValueError("No se encontraron columnas numéricas esperadas. Revisa los nombres de columnas del dataset.")

# 3. Diagnóstico inicial antes del ajuste.
print("\nTipos de datos antes del ajuste:")
print(df[columnas_numericas].dtypes)

print("\nNulos antes del ajuste:")
print(df[columnas_numericas].isnull().sum())

# 4. Conversión segura a formato numérico.
# errors='coerce' convierte valores no numéricos en NaN para poder tratarlos.
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nTipos de datos después de convertir a numérico:")
print(df[columnas_numericas].dtypes)

# 5. Imputación de nulos numéricos con la mediana.
# La mediana es más robusta ante valores extremos que la media.
print("\nImputación de valores nulos con la mediana:")
for col in columnas_numericas:
    nulos = df[col].isnull().sum()
    if nulos > 0:
        mediana = df[col].median()
        df[col] = df[col].fillna(mediana)
        print(f"{col}: {nulos} nulos reemplazados con mediana = {mediana}")
    else:
        print(f"{col}: sin nulos")

# 6. Validación de valores negativos donde no deberían existir.
columnas_sin_negativos = [
    "transaction_amount",
    "monthly_income",
    "customer_tenure_months",
    "avg_3m_transaction_amount",
    "avg_3m_transaction_count",
    "risk_score"
]

print("\nValidación de valores negativos:")
for col in columnas_sin_negativos:
    if col in df.columns:
        negativos = (df[col] < 0).sum()
        print(f"{col}: {negativos} valores negativos encontrados")

        if negativos > 0:
            mediana = df.loc[df[col] >= 0, col].median()
            df.loc[df[col] < 0, col] = mediana
            print(f"{col}: negativos reemplazados por la mediana de valores válidos = {mediana}")

# 7. Validación de columnas binarias.
columnas_binarias = ["anomaly_flag", "fraud_label"]

print("\nValidación de columnas binarias:")
for col in columnas_binarias:
    if col in df.columns:
        valores_unicos = sorted(df[col].dropna().unique())
        print(f"{col} - valores únicos antes de validar: {valores_unicos}")

        valores_invalidos = ~df[col].isin([0, 1])
        cantidad_invalidos = valores_invalidos.sum()

        if cantidad_invalidos > 0:
            moda = df.loc[df[col].isin([0, 1]), col].mode()
            moda = moda.iloc[0] if len(moda) > 0 else 0
            df.loc[valores_invalidos, col] = moda
            print(f"{col}: {cantidad_invalidos} valores inválidos reemplazados con la moda = {moda}")

# 8. Validación del rango lógico de risk_score.
if "risk_score" in df.columns:
    fuera_rango = ((df["risk_score"] < 0) | (df["risk_score"] > 100)).sum()
    print(f"\nrisk_score fuera del rango 0-100: {fuera_rango}")

    if fuera_rango > 0:
        df["risk_score"] = df["risk_score"].clip(lower=0, upper=100)
        print("risk_score ajustado al rango permitido entre 0 y 100.")

# 9. Resumen final.
print("\nNulos finales en columnas numéricas:")
print(df[columnas_numericas].isnull().sum())

print("\nResumen estadístico final de columnas numéricas:")
display(df[columnas_numericas].describe().T)

print("\nProceso de manejo/ajuste de columnas numéricas completado correctamente.")



5. MANEJO / AJUSTE DE COLUMNAS NUMÉRICAS - RONI
Dimensiones actuales del dataset: (593963, 31)

Columnas numéricas encontradas:
['transaction_amount', 'monthly_income', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count', 'deviation_from_usual_amount_pct', 'risk_score', 'anomaly_flag', 'fraud_label']

Tipos de datos antes del ajuste:
transaction_amount                 float64
monthly_income                     float64
customer_tenure_months               int64
avg_3m_transaction_amount          float64
avg_3m_transaction_count             int64
deviation_from_usual_amount_pct    float64
risk_score                           int64
anomaly_flag                         int64
fraud_label                          int64
dtype: object

Nulos antes del ajuste:
transaction_amount                 0
monthly_income                     0
customer_tenure_months             0
avg_3m_transaction_amount          0
avg_3m_transaction_count           0
deviation_from_usual_a

,count,mean,std,min,25%,50%,75%,max
transaction_amount,593963.0,33579.695934,278227.406703,50.00,1939.32,3565.715,7032.435,4999709.93
monthly_income,593963.0,61783.979203,45318.479726,7.08,30823.39,49434.920,82875.425,219782.40
customer_tenure_months,593963.0,61.994320,34.117396,3.00,32.00,62.000,92.000,120.00
avg_3m_transaction_amount,593963.0,4271.559408,3441.249572,245.14,1909.87,3266.070,5652.330,23041.93
avg_3m_transaction_count,593963.0,92.558523,84.041865,0.00,66.00,76.000,128.000,1500.00
deviation_from_usual_amount_pct,593963.0,212.526726,722.028988,-99.75,-20.70,4.920,33.900,6856.13
risk_score,593963.0,100.000000,0.000000,100.00,100.00,100.000,100.000,100.00
anomaly_flag,593963.0,0.130069,0.336379,0.00,0.00,0.000,0.000,1.00
fraud_label,593963.0,0.099733,0.299645,0.00,0.00,0.000,0.000,1.00



Proceso de manejo/ajuste de columnas numéricas completado correctamente.


#### 5. Proceso de manejo/ajuste de columnas numéricas (Roni)

El proceso de manejo y ajuste de columnas numéricas se realizó con el objetivo de asegurar que las variables cuantitativas del dataset financiero estuvieran correctamente formateadas, limpias y listas para las siguientes etapas del proyecto.

**Cómo se hizo:**  
Primero se identificaron las columnas numéricas esperadas dentro del dataset, tales como `transaction_amount`, `monthly_income`, `customer_tenure_months`, `avg_3m_transaction_amount`, `avg_3m_transaction_count`, `deviation_from_usual_amount_pct`, `risk_score`, `anomaly_flag` y `fraud_label`. Luego se validó cuáles de estas columnas existían realmente en el DataFrame para evitar errores por nombres no encontrados. Posteriormente, las columnas fueron convertidas a formato numérico utilizando `pd.to_numeric()` con `errors="coerce"`, lo que permitió transformar valores inválidos en valores nulos controlados. Después se imputaron los valores faltantes utilizando la mediana, se revisaron valores negativos en columnas donde no tienen sentido lógico y se validaron las columnas binarias para asegurar que solo contuvieran valores 0 y 1.

**Por qué se hizo:**  
Este ajuste fue necesario porque las columnas numéricas son fundamentales para el análisis financiero, la detección de anomalías y la construcción de modelos de Machine Learning. Si estas variables contienen textos, valores inválidos, nulos sin tratar, negativos incorrectos o formatos inconsistentes, pueden generar errores en los cálculos, afectar los análisis estadísticos y reducir la calidad de los modelos predictivos. Además, en un dataset financiero, variables como el monto de la transacción, el ingreso mensual, la antigüedad del cliente y el puntaje de riesgo deben mantener coherencia lógica.

**Para qué se hizo:**  
Este proceso permite dejar el dataset en mejores condiciones para etapas posteriores como la detección de outliers, el análisis exploratorio, la generación de reportes y el modelado predictivo de fraude. Al estandarizar y validar las columnas numéricas, se mejora la confiabilidad de los datos, se reducen errores de procesamiento y se garantiza que las variables utilizadas representen correctamente el comportamiento financiero de las transacciones.


#### Deteccion de outliers (Felix)

Realizamos un proceso de deteccion de outliers en los montos de transacciones, utilizando un metodo estadistico para identificar registros fuera del comportamiento normal.

Este analisis se llevo a cabo con el objetivo de mejorar la calidad de los datos, detectar posibles errores o anomalias y garantizar resultados mas confiables en los reportes y analisis posteriores.

In [75]:
print("\n" + "=" * 50)
print("DETECTANDO OUTLIERS")
print("=" * 50)

df["transaction_datetime"] = pd.to_datetime(df['transaction_datetime'],errors="coerce")

Q1 = df["transaction_amount"].quantile(0.25)
Q3 = df["transaction_amount"].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[
    (df["transaction_amount"] < limite_inferior) |
    (df["transaction_amount"] > limite_superior)
]

print(f"Cantidad de outliers detectados: {outliers.shape[0]}")


DETECTANDO OUTLIERS
Cantidad de outliers detectados: 65748



### 1. Proceso de los duplicados. (Arlene)
### 2. Proceso de ajuste/correcion de fechas/horas (Daniel)
### 3. Proceso de manejo de datos nulos (Adison)
### 4. Proceso de homogenizar de texto (Adison)
### 5. Proceso de manejo/ajuste de columnas numericas (Roni)
### 6. Deteccion de outliers (Felix)
### 7. Repo de Github (Daniel)
### Disclaimer de la bitacora de CADA PUNTO, haganlo como un registro detallado. 3 puntos a responder: como, por que y para que